In [1]:
import os

# 1. Kaggle API Credentials
os.environ['KAGGLE_USERNAME'] = "kemal004"
os.environ['KAGGLE_KEY'] = "KGAT_36d73add01697d80b196b38359b89dc9"

# 2. Download and Unzip the Dataset
!kaggle datasets download -d ammarnassanalhajali/pklot-dataset
!unzip -q pklot-dataset.zip -d pklot_data

Dataset URL: https://www.kaggle.com/datasets/ammarnassanalhajali/pklot-dataset
License(s): unknown
100% 843M/843M [00:06<00:00, 143MB/s]



In [2]:
import json
import os

def coco_to_yolo(json_path, output_dir):
    if not os.path.exists(json_path):
        return

    with open(json_path, 'r') as f:
        data = json.load(f)

    # Map COCO categories to YOLO indices (0: empty, 1: occupied)
    cat_map = {}
    for cat in data['categories']:
        name = cat['name'].lower()
        if 'empty' in name or 'space-empty' in name:
            cat_map[cat['id']] = 0
        else:
            cat_map[cat['id']] = 1  # Occupied

    # Format conversion
    img_map = {img['id']: img for img in data['images']}
    yolo_annotations = {img['id']: [] for img in data['images']}

    for ann in data['annotations']:
        img_id = ann['image_id']
        cat_id = ann['category_id']

        if cat_id not in cat_map: continue
        yolo_class = cat_map[cat_id]

        bbox = ann['bbox']
        img_info = img_map[img_id]
        img_w, img_h = img_info['width'], img_info['height']

        # YOLO Normalization (x_center, y_center, width, height)
        x_center = (bbox[0] + bbox[2] / 2) / img_w
        y_center = (bbox[1] + bbox[3] / 2) / img_h
        w = bbox[2] / img_w
        h = bbox[3] / img_h

        yolo_annotations[img_id].append(f"{yolo_class} {x_center:.6f} {y_center:.6f} {w:.6f} {h:.6f}")

    count = 0
    for img_id, lines in yolo_annotations.items():
        img_info = img_map[img_id]
        txt_name = os.path.splitext(img_info['file_name'])[0] + '.txt'
        txt_path = os.path.join(output_dir, txt_name)

        with open(txt_path, 'w') as f:
            f.write('\n'.join(lines))
        count += 1

    print(f"✅ Generated {count} YOLO label (.txt) files in {output_dir}")

coco_to_yolo('/content/pklot_data/train/_annotations.coco.json', '/content/pklot_data/train')
coco_to_yolo('/content/pklot_data/valid/_annotations.coco.json', '/content/pklot_data/valid')
coco_to_yolo('/content/pklot_data/test/_annotations.coco.json', '/content/pklot_data/test')

✅ Generated 8691 YOLO label (.txt) files in /content/pklot_data/train
✅ Generated 2483 YOLO label (.txt) files in /content/pklot_data/valid
✅ Generated 1242 YOLO label (.txt) files in /content/pklot_data/test


In [3]:
import yaml
import os

data_yaml_content = {
    'train': '/content/pklot_data/train',
    'val': '/content/pklot_data/valid',
    'test': '/content/pklot_data/test',
    'nc': 2,
    'names': ['empty', 'occupied']
}

data_yaml_path = '/content/pklot_data/data.yaml'

with open(data_yaml_path, 'w') as f:
    yaml.dump(data_yaml_content, f, default_flow_style=False)

In [4]:
!pip install ultralytics
from ultralytics import YOLO

# Load the nano model for speed and efficiency
model = YOLO('yolov8n.pt')

# Train the model
results = model.train(
    data='/content/pklot_data/data.yaml',
    epochs=3,
    imgsz=640,
    batch=16,
    project='Smart_Parking',
    name='MVP_Model_Final'
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 31.6 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.30 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/pklot_data/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=3, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.

In [5]:
from google.colab import files

files.download('/content/runs/detect/Smart_Parking/MVP_Model_Final/weights/best.pt')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>